# 01. Data Overview

## Prerequisites

This notebook gives a first look at the data used in `transferformance`, including its stucture, coverage, and quality.

### Data Origin

The main data source is [transfermarkt-datasets](https://github.com/dcaribou/transfermarkt-datasets), created and maintained by [David Cereijo](https://github.com/dcaribou), It collects, processes, and catalogues data from the [Transfermarkt website](https://www.transfermarkt.com/), and is distributed in many forms. This project will be using the `.duckdb` database stored in an [author-maintained Cloudflare R2 Public bucket](https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb).

`transfermarkt-datasets` is made available under the Creative Commons CC0 1.0 Universal Public Domain Dedication.

### Data Description

*According to `dcaribou/transfermarkt-datasets`'s `README`:*

> The dataset is composed of 12 tables covering competitions, games, clubs, players, appearances, player valuations, club games, game events, game lineups, transfers, countries and national teams. Each table contains the attributes of the entity and IDs that can be used to join them together.

### Caveats

According to the `transfermarkt-datasets`'s Github Repo, updates are paused for the time being. The data is current to July 6th, 2026; which includes the FIFA World Cup 2026's Group Stage.

### Decisions

The dataset has 12 tables, but a valuation model doesn't need all of them. To keep the project focused and the feature set small, we selected tables that helps explain a player's market value at any given point in time.

- **Yes:** core tables for the target and features.
- **Optional:** potentially useful, but only added if leakage can be ruled out.
- **No:** out of scope for this version.

| Table | Description | Used | Reason | 
| --- | --- | --- | --- |
| `players` | Player profiles, positions, market values, international caps | Yes | Core part of the project, contains players' static attributes |
| `player_valuations` | Historical market value records | Yes | Core part of the project, target variable |
| `appearances` | One row per player per game played | Yes | Players' stats by season |
| `clubs` | Club details, squad size, market value | Yes | Add clubs and leagues context to the player's valuation |
| `competitions` | Leagues, tournaments and national team competitions | Optional | Use to decide the level of leagues and clubs |
| `transfers` | Player transfers between clubs | Optional | Provide valuable insight about players' actual valuations, but has potential leakage problems | 
| `games` | Match results, lineups, attendance | No | Out of scope |
| `club_games` | Per-club view of each game | No | Out of scope |
| `game_events` | Goals, cards, substitutions | No | Out of scope |
| `game_lineups` | Starting and bench lineups | No | Out of scope |
| `countries` | Country details and confederation membership | No | Not needed |
| `national_teams` | National team profiles, squad size, FIFA ranking | No | Not needed |

## Preperation

### Import libraires

In [37]:
from transferformance.configs import PATH_RAW_DATA

import duckdb

### Import the dataset

In [38]:
con = duckdb.connect(PATH_RAW_DATA, read_only=True)

## Data overview

### `player_valuations` table

#### Basic observations

First, we check out the `player_valuations` table structure and its basic statistics.

In [39]:
print("`player_valuations`'s table structure:")

con.sql("""
    DESCRIBE player_valuations
""").show()

print("`player_valuations`'s table basic statistics:")

con.sql("""
    SUMMARIZE player_valuations
""").show()

`player_valuations`'s table structure:
┌─────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│             column_name             │ column_type │  null   │   key   │ default │  extra  │
│               varchar               │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id                           │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ date                                │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ market_value_in_eur                 │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_name                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_id                     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_club_domestic_competition_id │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────

In [40]:
print("Top 10 player valuations, sorted by `market_value_in_eur` descending.")

con.sql("""
    SELECT *
    FROM player_valuations v
        JOIN LATERAL (
            SELECT p.name
            FROM players p
            WHERE v.player_id = p.player_id
        ) ON TRUE
    ORDER BY market_value_in_eur DESC
    LIMIT 10 
""").show()

Top 10 player valuations, sorted by `market_value_in_eur` descending.
┌───────────┬────────────┬─────────────────────┬───────────────────┬─────────────────┬─────────────────────────────────────┬─────────────────┐
│ player_id │    date    │ market_value_in_eur │ current_club_name │ current_club_id │ player_club_domestic_competition_id │      name       │
│   int32   │    date    │        int32        │      varchar      │      int32      │               varchar               │     varchar     │
├───────────┼────────────┼─────────────────────┼───────────────────┼─────────────────┼─────────────────────────────────────┼─────────────────┤
│    342229 │ 2025-12-12 │           200000000 │ Real Madrid       │             418 │ ES1                                 │ Kylian Mbappé   │
│    937958 │ 2026-06-05 │           200000000 │ FC Barcelona      │             131 │ ES1                                 │ Lamine Yamal    │
│    371998 │ 2024-10-11 │           200000000 │ Real Madrid       │    

**Initial observations:**
- The `player_valuations` table consists of 6 columns: `player_id`, `date`, `market_value_in_eur`, `current_club_name`, `current_club_id`, and `player_club_domestic_competition_id`; having 656,301 entries for 42,892 distinct players. Player valuations span 2000-01-20 to 2026-06-12, which matches the snapshot date noted earlier.
- All columns in the table are completed excepts for `player_club_domestic_competition_id`, which has a 14.17% null value rate.
- Despite having only 5,378 `current_club_id`, there are 8,119 `current_club_name`.
- `market_value_in_eur`seems to be heavily right-skewed: the mean (average) of approximately 2.2 million euros is much higher than the median (q50) of only about 500,000 euro.
- The minimum `market_value_in_eur` is 0 euros.
- Only 33 domestic leagues were covered in the table, showing that the coverage is limited and focused on bigger leagues.

**Follow-ups:**
- The number of current club ids is lesser than the number of club names. Possible causes: the club id and the club name is not sync, or the same club id is used for different teams (B-teams, youth squads, club renames, etc.)
- `player_club_domestic_competition_id` is missing in 14.17% of rows even though `current_club_id` and `current_club_name` are completed. Possible causes: clubs outside the covered leagues, placeholder club values (e.g. "Without Club", "Retired"), or a mapping that was never updated. Check which clubs the nulls belong to.
- `market_value_in_eur` has a minimum of 0. Possible causes: placeholder values, retired players, or players without a club. Check how many zeros exist and where they occur.
- Check for duplicate (`player_id`, `date`) keys and for valuations whose `player_id` is missing from `players`.

#### Duplicate (`player_id`, `date`) keys check.

First, we check whether all (`player_id`, `date`) pair are unique.

In [41]:
con.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (player_id, date)) AS total_distinct_keys
    FROM player_valuations
""").show()

┌────────────┬─────────────────────┐
│ total_rows │ total_distinct_keys │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│     656301 │              656301 │
└────────────┴─────────────────────┘



**Observation:** The table has 656,301 rows and 656,301 distinct (`player_id`, `date`) pairs, so there are no duplicate keys. Each player has at most one valuation per date.

**Decision:** No deduplication needed.

Second, we check whether any valuation belongs to a player that does not exist in the `players` table.

In [42]:
con.sql("""
    SELECT 
        COUNT(*) AS orphan_rows,
        COUNT(DISTINCT v.player_id) AS orphan_players
    FROM player_valuations v
    LEFT JOIN players p ON p.player_id = v.player_id
    WHERE p.player_id IS NULL
""").show()

┌─────────────┬────────────────┐
│ orphan_rows │ orphan_players │
│    int64    │     int64      │
├─────────────┼────────────────┤
│           0 │              0 │
└─────────────┴────────────────┘



**Observation:** All valuations match a player in `players`: there are 0 orphan rows and 0 orphan players.

**Decision:** No rows dropped. `player_valuations` can be joined to `players` on `player_id` without losing data.

**Observations:**

#### `current_club_name`, `current_club_id`, `player_club_domestic_competition_id` discrepancies

First, we check the amount of different club names per club id.

In [43]:
con.sql("""
    SELECT
        v.current_club_id,
        c.name AS official_name,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT v.current_club_name) AS n_names,
        list(DISTINCT v.current_club_name)[1:10] AS sample_names
    FROM player_valuations AS v
    LEFT JOIN clubs AS c
        ON c.club_id = v.current_club_id
    GROUP BY v.current_club_id, c.name
    ORDER BY n_names DESC
    LIMIT 10
""").show()

┌─────────────────┬──────────────────────────────┬────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ current_club_id │        official_name         │ n_rows │ n_names │                                                                                               sample_names                                                                                               │
│      int32      │           varchar            │ int64  │  int64  │                                                                                                varchar[]                                                                                                 │
├─────────────────┼──────────────────────────────┼────────┼─────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────

**Observations:** Contrary to our hypothesis that the gap comes from club renames, a single `current_club_id` is attached to many unrelated clubs. For example, the ID that `clubs` maps to `APO Levadiakos Football Club` appears in `player_valuations` with 400 different `current_club_name` values.

**Possible reasons:** the ID and the name may not describe the same point in time (e.g. the name reflects the club at the valuation date while the ID is filled from a different source or snapshot), or the ID may have been assigned incorrectly during data processing. We test this next by inspecting individual valuation histories.

After inspecting several players with `current_club_id = 2672`, we select player with ID `94318` (Jovan Krneta) as a clear example of the pattern. In the player's full valuation history, the club ID stays at 2672 while `current_club_name` changes between unrelated clubs.

In [44]:
con.sql("""
    SELECT 
        p.name AS "player_name", 
        v.date, 
        v.current_club_id, 
        v.current_club_name, 
        c.name AS "actual_club_name",
        v.player_club_domestic_competition_id
    FROM player_valuations v
        JOIN LATERAL (
            SELECT p.name
            FROM players p
            WHERE p.player_id = v.player_id
        ) p ON TRUE
        JOIN LATERAL (
            SELECT c.name
            FROM clubs c
            WHERE c.club_id = v.current_club_id
        ) c ON TRUE
    WHERE v.player_id = 94318
    ORDER BY v.date
""").show(max_rows=40)

┌──────────────┬────────────┬─────────────────┬─────────────────────────────┬──────────────────────────────┬─────────────────────────────────────┐
│ player_name  │    date    │ current_club_id │      current_club_name      │       actual_club_name       │ player_club_domestic_competition_id │
│   varchar    │    date    │      int32      │           varchar           │           varchar            │               varchar               │
├──────────────┼────────────┼─────────────────┼─────────────────────────────┼──────────────────────────────┼─────────────────────────────────────┤
│ Jovan Krneta │ 2010-03-28 │            2672 │ Unknown                     │ APO Levadiakos Football Club │ GR1                                 │
│ Jovan Krneta │ 2010-04-19 │            2672 │ Unknown                     │ APO Levadiakos Football Club │ GR1                                 │
│ Jovan Krneta │ 2012-04-09 │            2672 │ Red Star Belgrade           │ APO Levadiakos Football Club │ GR1      

We now need to check what percetange of player valuations does this phenomenon applies to.

In [45]:
# Is the club ID constant within each player's history?
con.sql("""
    WITH career_track AS (
        SELECT player_id,
               COUNT(DISTINCT current_club_id) AS n_ids,
               COUNT(DISTINCT current_club_name) AS n_names
        FROM player_valuations
        GROUP BY player_id
    )
    SELECT
        COUNT(*) AS n_players,
        SUM((n_ids = 1)::INT) AS one_id,
        SUM((n_ids > 1)::INT) AS multiple_ids,
        SUM((n_ids = 1 AND n_names > 1)::INT) AS one_id_wrong_names,
        SUM((n_ids > 1 AND n_names > 1 AND n_ids <> n_names)::INT) AS mutiple_id_wrong_names,
        SUM((n_ids = n_names)::INT) AS equal_id_counts
    FROM career_track
""").show()

┌───────────┬────────┬──────────────┬────────────────────┬────────────────────────┬─────────────────┐
│ n_players │ one_id │ multiple_ids │ one_id_wrong_names │ mutiple_id_wrong_names │ equal_id_counts │
│   int64   │ int128 │    int128    │       int128       │         int128         │     int128      │
├───────────┼────────┼──────────────┼────────────────────┼────────────────────────┼─────────────────┤
│     41528 │  22707 │        18821 │              17674 │                   1535 │           22313 │
└───────────┴────────┴──────────────┴────────────────────┴────────────────────────┴─────────────────┘



**Observations:** We can see that out of 41,528 players with valuations:
- 17,674 players (42.6%) have their `current_club_id` despite their `current_club_name` changing throughout their career.
- 1,523 players (3.7%) have their `current_club_id` count not maching 1:1 with their `current_club_name`.
- 22,313 players (53.7%) have equal `current_club_id` and `current_club_name` count.

Similarly, looking at the Jovan Krneta career table, we can see that their `player_club_domestic_competition_id` was unchanged (`GR1`, Greece Super League) despite their club progression (`Red Star Belgrade` is a club that plays in Serbia's top domestic league, not Greece's).

In [46]:
con.sql("""
    SELECT competition_id, name, country_name, type
    FROM competitions 
    WHERE competition_id = 'GR1'
""").show()

con.sql("""
    SELECT 
        c.name AS "club_name", 
        c.domestic_competition_id, 
        l.name AS "competition_name",
        l.country_name,
        l.type AS "competition_type"
    FROM clubs c
        JOIN LATERAL (
            SELECT cm.name, cm.country_name, cm.type
            FROM competitions cm
            WHERE cm.competition_id = c.domestic_competition_id
        ) l ON TRUE
    WHERE c.name = 'Red Star Belgrade'
""").show()

┌────────────────┬────────────────┬──────────────┬─────────────────┐
│ competition_id │      name      │ country_name │      type       │
│    varchar     │    varchar     │   varchar    │     varchar     │
├────────────────┼────────────────┼──────────────┼─────────────────┤
│ GR1            │ super-league-1 │ Greece       │ domestic_league │
└────────────────┴────────────────┴──────────────┴─────────────────┘

┌───────────────────┬─────────────────────────┬───────────────────┬──────────────┬──────────────────┐
│     club_name     │ domestic_competition_id │ competition_name  │ country_name │ competition_type │
│      varchar      │         varchar         │      varchar      │   varchar    │     varchar      │
├───────────────────┼─────────────────────────┼───────────────────┼──────────────┼──────────────────┤
│ Red Star Belgrade │ SER1                    │ super-liga-srbije │ Serbia       │ domestic_league  │
└───────────────────┴─────────────────────────┴───────────────────┴────────

From these examples and findings, we can conclude that the three columns `current_club_id`, `current_club_name` and `player_club_domestic_competition_id` are unreliable, with nearly half of the players with valuations having mismatched club and league information.

**Decision:** 
- `current_club_id`, `current_club_name`, and `player_club_domestic_competition_id` are excluded from the feature set. Club and league context, if added later, will be built from another source (e.g. `appearances`).
- The remaining columns (`player_id`, `date`, `market_value_in_eur`) are kept.

#### Irregular `market_value_in_eur` minimum value

From the basic observations, we found out that the minimum player valuation is 0, which is irregular for a table about player valuation. 

By checking the frequency of the zero valuations, we can determine whether it is a placeholder value (as opposed to not having a valuation at all). 

In [47]:
print("Number of player valuations with `market_value_in_eur` = 0")
con.sql("""
    SELECT COUNT(*) AS "n_zero_valuations"
    FROM player_valuations
    WHERE market_value_in_eur = 0
""").show()

print("Players with `market_value_in_eur` = 0")
con.sql("""
    SELECT *
    FROM player_valuations v
    JOIN LATERAL (
        SELECT p.name
        FROM players p
        WHERE p.player_id = v.player_id
    ) l ON TRUE
    WHERE market_value_in_eur = 0
""").show()


print("Full player valuation log of Miral Samardzic")
con.sql("""
    SELECT *
    FROM player_valuations v
    WHERE player_id = 60096
""").show()


Number of player valuations with `market_value_in_eur` = 0
┌───────────────────┐
│ n_zero_valuations │
│       int64       │
├───────────────────┤
│                 1 │
└───────────────────┘

Players with `market_value_in_eur` = 0
┌───────────┬────────────┬─────────────────────┬───────────────────┬─────────────────┬─────────────────────────────────────┬─────────────────┐
│ player_id │    date    │ market_value_in_eur │ current_club_name │ current_club_id │ player_club_domestic_competition_id │      name       │
│   int32   │    date    │        int32        │      varchar      │      int32      │               varchar               │     varchar     │
├───────────┼────────────┼─────────────────────┼───────────────────┼─────────────────┼─────────────────────────────────────┼─────────────────┤
│     60096 │ 2008-03-17 │                   0 │ NK Maribor        │            2696 │ RU1                                 │ Miral Samardzic │
└───────────┴────────────┴─────────────────────┴──────

**Observations:** There is only 1 player valuation with `market_value_in_eur` equal to zero, belonging to Miral Samardzic. After checking Samardzic's full valuation history, the zero valuation (February 17th, 2008) was most likely a data entry mistake, since the next valuation, on February 18th, 2008, was 200,000 euros.

**Decision:** Removing the entry. The value is irregular, may affect the analysis and model training, and has negligable effect if removed (1 removed out of 656,301 rows).

### `players` table

#### Basic observations

First, we check the table's stucture and basic statistics.

In [48]:
print("`players`'s table structure:")

display(    
    con.sql("""
        DESCRIBE players
    """).df()
)

`players`'s table structure:


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,first_name,VARCHAR,YES,None,None,None
2,last_name,VARCHAR,YES,None,None,None
3,name,VARCHAR,YES,None,None,None
4,last_season,VARCHAR,YES,None,None,None
5,current_club_id,VARCHAR,YES,None,None,None
6,player_code,VARCHAR,YES,None,None,None
7,country_of_birth,VARCHAR,YES,None,None,None
8,city_of_birth,VARCHAR,YES,None,None,None
9,country_of_citizenship,VARCHAR,YES,None,None,None


In [49]:
print("`players`'s table basic statistics:")

display(
    con.sql("""
        SUMMARIZE players
    """).df().sort_values(by="null_percentage", ascending=False)
)

`players`'s table basic statistics:


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
20,current_national_team_id,VARCHAR,10521,9173,120,NaN,NaN,NaN,NaN,NaN,50149,93.50
18,international_caps,INTEGER,0,233,161,11.370737755734655,18.988176426730625,1,4,12,50149,61.40
19,international_goals,INTEGER,0,146,70,1.182940690225253,4.159747368130747,0,0,1,50149,61.40
16,agent_name,VARCHAR,#LEADERS,Öner Karalar,3958,NaN,NaN,NaN,NaN,NaN,50149,46.45
15,contract_expiration_date,TIMESTAMP,2000-05-31 00:00:00,2035-06-30 00:00:00,243,2026-11-01 02:12:52.438252,NaN,2025-02-28 22:17:12.287181,2027-01-03 23:48:22.675868,2028-06-30 00:00:00,50149,37.03
25,highest_market_value_in_eur,INTEGER,10000,200000000,210,3450773.4540550956,9347394.390237499,292408,758719,2501512,50149,17.19
24,market_value_in_eur,INTEGER,10000,200000000,123,1511049.2920439222,5854556.454154651,102050,265673,721790,50149,17.19
13,foot,VARCHAR,both,right,3,NaN,NaN,NaN,NaN,NaN,50149,11.75
7,country_of_birth,VARCHAR,Afghanistan,Zimbabwe,216,NaN,NaN,NaN,NaN,NaN,50149,11.74
8,city_of_birth,VARCHAR,'s-Hertogenbosch,"—, Kumamoto",12118,NaN,NaN,NaN,NaN,NaN,50149,11.18


**Initial observations:**
- `players` has 26 columns and 50,149 rows. `player_id` has no nulls.
- This table has more players than `player_valuations` (50,149 vs. 41,528), so around 8,600 players probably have no valuation history.
- About snapshot-date values:
    - `market_value_in_eur` and `highest_market_value_in_eur` are both 17.19% null. They are snapshot values (the current and peak valuation), so they must not be used as features, since they would leak the target.
    - Several columns describe the player at the snapshot date, not at the time of each valuation: `current_club_id`, `current_club_name`, `current_club_domestic_competition_id`, `contract_expiration_date` (37% null), `international_caps` and `international_goals` (61.4% null), `current_national_team_id` (93.5% null) and `last_season`.
- Usable candidates are: 
    - `date_of_birth` (0.1% null, to compute age at each valuation date)
    - `position` (no nulls), `sub_position` (1.17% null). `position` includes a "Missing" category.
    - `height_in_cm` (8.7% null) and `foot` (11.75% null). `height_in_cm` has a minimum of 17 cm, which is not plausible.
- `agent_name` (46% null), names, `player_code`, `url` and `image_url` are identifiers or free text with no predictive use.

**Follow-ups:**
- Confirm that `player_id` is unique.
- Check whether the 17.19% null `market_value_in_eur` corresponds to players with no valuations.
- Count the implausible heights and the "Missing" positions, then decide whether to drop or impute them.

#### `player_id` being unique

We check whether all `player_id` values are unique.

In [50]:
con.sql("""
    SELECT
        COUNT(*) AS n_rows, 
        COUNT(DISTINCT player_id) AS n_ids 
    FROM players
""").show()

┌────────┬───────┐
│ n_rows │ n_ids │
│ int64  │ int64 │
├────────┼───────┤
│  50149 │ 50149 │
└────────┴───────┘



**Conclusion:** Because each `player_id` is unique, we can say that each row in `players` represents a different player.

#### `market_value_in_eur` corresponding to players with no valuations 

First, we check if every players with no `market_value_in_eur` also have no player valuations in `player_valuations` (and vice versa).

In [51]:
con.sql("""
    SELECT 
        (v.player_id IS NULL) AS no_valuations,
        (p.market_value_in_eur IS NULL) AS null_current_value,
        COUNT(*) AS n
    FROM players p
    LEFT JOIN (
        SELECT DISTINCT player_id 
        FROM player_valuations
    ) v ON p.player_id = v.player_id
    GROUP BY 1, 2
""").show()

┌───────────────┬────────────────────┬───────┐
│ no_valuations │ null_current_value │   n   │
│    boolean    │      boolean       │ int64 │
├───────────────┼────────────────────┼───────┤
│ true          │ true               │  8621 │
│ false         │ false              │ 41528 │
└───────────────┴────────────────────┴───────┘



**Observation:** Out of 50,149 players, all of them fall into one of two groups:
- 41,528 (82.8%) have at least one valuation in `player_valuations` and a non-null `market_value_in_eur` in `players`.
- 8,621 (17.2%) have no valuations at all and a null `market_value_in_eur`.

There are no players with valuations but a missing current value, or vice versa. This explains the 17.19% null rate in `market_value_in_eur`.

**Decision:** The 8,621 players without valuations have no target, so they are excluded.


Next, we check whether `market_value_in_eur` in `players` equals each player's latest valuation in `player_valuations`. Similarly, we check whether `highest_market_value_in_eur` equals each player's highest valuation to date.

In [52]:
print("Comparision between latest valuation in `player_valuation` with `players`'s `market_value_in_eur`")

con.sql("""
    WITH latest_valuations AS (
        SELECT DISTINCT ON (player_id)
            player_id, 
            market_value_in_eur AS latest_value
        FROM player_valuations
        ORDER BY player_id, date DESC
    )
    SELECT
        COUNT(*) AS "n_valuations",
        SUM(p.market_value_in_eur = l.latest_value) AS "n_synced"
    FROM players p JOIN latest_valuations l USING (player_id)
""").show()

print("Comparision between highest valuation in `player_valuation` with `players`'s `highest_market_value_in_eur`")

con.sql("""
    WITH latest_valuations AS (
        SELECT DISTINCT ON (player_id)
            player_id, 
            market_value_in_eur AS highest_value
        FROM player_valuations
        ORDER BY player_id, market_value_in_eur DESC
    )
    SELECT
        COUNT(*) AS "n_valuations",
        SUM(p.highest_market_value_in_eur = l.highest_value) AS "n_synced"
    FROM players p JOIN latest_valuations l USING (player_id)
""").show()

Comparision between latest valuation in `player_valuation` with `players`'s `market_value_in_eur`
┌──────────────┬──────────┐
│ n_valuations │ n_synced │
│    int64     │  int128  │
├──────────────┼──────────┤
│        41528 │    41528 │
└──────────────┴──────────┘

Comparision between highest valuation in `player_valuation` with `players`'s `highest_market_value_in_eur`
┌──────────────┬──────────┐
│ n_valuations │ n_synced │
│    int64     │  int128  │
├──────────────┼──────────┤
│        41528 │    41528 │
└──────────────┴──────────┘



**Observation:** For all 41,528 players with valuations: 
- `players.market_value_in_eur` equals their latest valuation in `player_valuations` (100% match).
- `players.highest_market_value_in_eur` equals their highest valuation to date (100% match).

**Decision:** Both columns are derived from the same valuation history we want to predict, so they carry no independent information. Using them as features would leak the target (the latest and peak values of the same series). They are excluded from the feature set.

#### Players' personal data

We now examine the columns that describes the player as a person (date of birth, height, preferred foot, nationality and birthplace). These are the main candidates for static features, so we check their completeness and whether their values are plausible.

##### Players' `height_in_cm`

In the basic statistics, the minimum `height_in_cm` was 17 cm, which is implausible. We therefore check whether other players have similarly irregular heights.

For professional football players, a range of roughly 150 to 210 cm is plausible, so we treat values outside it as suspicious.

In [53]:
con.sql("""
    SELECT height_in_cm, COUNT(*) AS n_players
    FROM players
    WHERE height_in_cm NOT BETWEEN 150 AND 210
    GROUP BY 1 
    ORDER BY 1
""").show()

┌──────────────┬───────────┐
│ height_in_cm │ n_players │
│    int32     │   int64   │
├──────────────┼───────────┤
│           17 │         2 │
│           18 │         9 │
│           19 │         2 │
└──────────────┴───────────┘



**Observation:** Only 13 players (0.03% of 50,149) have a `height_in_cm` outside 150 to 210 cm: 2 at 17 cm, 9 at 18 cm, and 2 at 19 cm. All are physically impossible for adult players, so they look like data entry errors. No heights above 210 cm were found.

**Decision:** Set these 13 values to null instead of dropping the players, since their other attributes are still valid. They are then handled together with the other missing heights (8.7% null) by imputing the median height of the player's position.

##### Players' `date_of_birth` and age

We now examine players' `date_of_birth`. From the basic statistics, players with a known date of birth were born between July 31st, 1968 and February 23rd, 2011, and 0.1% of the values are null.

First, we compute each player's age at the time of each valuation in `player_valuations` and look at the distribution.

In [54]:
con.sql("""
    SELECT 
        DATE_SUB('YEAR', p.date_of_birth, v.date) AS age,
        COUNT(*) AS n_valuations
    FROM player_valuations v
        JOIN players p USING (player_id)
    WHERE p.date_of_birth IS NOT NULL
    GROUP BY age
""").show()


┌───────┬──────────────┐
│  age  │ n_valuations │
│ int64 │    int64     │
├───────┼──────────────┤
│     2 │            1 │
│     3 │            1 │
│    11 │            1 │
│    13 │            3 │
│    14 │            6 │
│    15 │           76 │
│    16 │         2501 │
│    17 │        10441 │
│    18 │        25204 │
│    19 │        41389 │
│     · │           ·  │
│     · │           ·  │
│     · │           ·  │
│    36 │         4303 │
│    37 │         2528 │
│    38 │         1356 │
│    39 │          686 │
│    40 │          331 │
│    41 │          158 │
│    42 │           58 │
│    43 │           17 │
│    44 │            4 │
│    45 │            3 │
└───────┴──────────────┘
  36 rows    2 columns
  (20 shown)           



**Observations:**
- Almost all valuations fall at plausible ages for professional footballers. The oldest valuation is at age 45 (3 valuations), and counts taper off steadily after the mid-30s.
- 12 valuations imply an age below 15: 1 at age 2, 1 at age 3, 1 at age 11, 3 at age 13 and 6 at age 14. Ages 2, 3 and 11 are impossible for valued players, so either the date of birth or the valuation date must be wrong. Ages 13 and 14 are very unlikely.
- Ages 15 and 16 are uncommon but plausible (76 valuations at 15, 2,501 at 16), since top youth prospects are valued early.

**Decision:**
- Valuations without a date of birth (600 rows) are dropped, since age at the valuation date is a core feature and cannot be reconstructed.
- Valuations with an age below 15 (12 rows, about 0.002% of the data) are dropped as likely data errors.
- Valuations at ages 15 to 45 are kept.

##### Players' `position` and `sub_position`

Moving on to players' on-field positions, we found that `position` has no null values but contains a `Missing` label, while `sub_position` has a 1.17% null rate. We now check whether these two facts are related, and how the two columns are structured.

In [55]:
con.sql("""
    SELECT 
        position, 
        sub_position, 
        COUNT(*) AS n_players
    FROM players 
    GROUP BY 1, 2
    ORDER BY 1, n_players DESC
""").show(max_rows=40)

┌────────────┬────────────────────┬───────────┐
│  position  │    sub_position    │ n_players │
│  varchar   │      varchar       │   int64   │
├────────────┼────────────────────┼───────────┤
│ Attack     │ Centre-Forward     │      6749 │
│ Attack     │ Left Winger        │      3297 │
│ Attack     │ Right Winger       │      3170 │
│ Attack     │ Second Striker     │       286 │
│ Defender   │ Centre-Back        │      8766 │
│ Defender   │ Right-Back         │      3700 │
│ Defender   │ Left-Back          │      3495 │
│ Goalkeeper │ Goalkeeper         │      5828 │
│ Midfield   │ Central Midfield   │      5498 │
│ Midfield   │ Defensive Midfield │      4123 │
│ Midfield   │ Attacking Midfield │      3520 │
│ Midfield   │ Left Midfield      │       566 │
│ Midfield   │ Right Midfield     │       565 │
│ Missing    │ NULL               │       586 │
└────────────┴────────────────────┴───────────┘
  14 rows                           3 columns



Next, for players with no position (`Missing`), we check whether their valuation data differ from the rest.

In [56]:
con.sql("""
    SELECT
        (p.position = 'Missing')       AS position_missing,
        COUNT(DISTINCT p.player_id)    AS n_players,
        COUNT(*)                       AS n_valuations,
        MEDIAN(v.market_value_in_eur)  AS median_value
    FROM player_valuations v
    JOIN players p USING (player_id)
    GROUP BY 1
""").show()

┌──────────────────┬───────────┬──────────────┬──────────────┐
│ position_missing │ n_players │ n_valuations │ median_value │
│     boolean      │   int64   │    int64     │    double    │
├──────────────────┼───────────┼──────────────┼──────────────┤
│ true             │       129 │          891 │     200000.0 │
│ false            │     41399 │       655410 │     500000.0 │
└──────────────────┴───────────┴──────────────┴──────────────┘



**Observations:**
- About the players with `sub_position`
    - `sub_position` is nested within `position`: each of the 13 sub-positions belongs to exactly one of the four positions (Attack: 4, Defender: 3, Midfield: 5, Goalkeeper: 1).
    the reverse.
    - Goalkeepers have a single sub-position, so for them the two columns are identical.
- About the players without `sub_position`
    - The `Missing` label in `position` and the nulls in `sub_position` are the same 586 players (1.17% of 50,149). No player has a position but no sub-position, or 
    - Of the 586 `Missing` players, only 129 have valuations. They account for 0.31% of the 41,528 modeling-set players and 891 of 656,301 valuations (0.14%).
    - Their median valuation is 200,000 €, against 500,000 € for the other players. This suggests a lower-value group, but it rests on a small sample.

**Decision:** 
- The 129 `Missing` players (891 valuations) are dropped, since their position is unknown and they cannot be assigned to any of the four positions. The impact on the dataset is negligible. 
- `position` (4 categories) is used as the position feature. `sub_position` is excluded from this version because it is nested in `position`, adds 13 categories, and is null for exactly the same players.

##### Players' `foot`

We now look deeper into players' dominant foot.

From the basic statistics, we saw that the column contains three distinct labels: `right`, `left`, and `both`, while also having a 11.75% null value rate.

In [57]:
con.sql("""
    SELECT
        foot,
        COUNT(*) AS n_players
    FROM players
    GROUP BY foot
""").show()

┌─────────┬───────────┐
│  foot   │ n_players │
│ varchar │   int64   │
├─────────┼───────────┤
│ right   │     31445 │
│ left    │     11000 │
│ both    │      1811 │
│ NULL    │      5893 │
└─────────┴───────────┘



**Observation:**
- `foot` has only the three expected labels: `right` (31,445; 62.7%), `left` (11,000; 21.9%) and `both` (1,811; 3.6%). 
- The remaining 5,893 players (11.75%) has no dominant foot information.
- Over 85% of players with a known foot are right- or left-footed, and `both` is rare.

**Decision:** Keep `foot` as a categorical feature and encode the nulls as an explicit `unknown` label.

##### Other features

**Decision:** `country_of_citizenship`, `country_of_birth` and `city_of_birth` are excluded from the feature set and are not imputed.
- High cardinality: about 208 citizenships, 216 birth countries and over 12,000 birth cities. Encoding these would add many sparse features and a high risk of overfitting for little gain.
- Missing values: `country_of_birth` and `city_of_birth` are about 11% nulls, so they would also need imputing, which adds complexity.
- Overlapping information: birthplace largely overlaps with citizenship, and neither is available without extra grouping work (e.g. by confederation).

### `appearances` table

#### Basic observations

First, we check the table's stucture and basic statistics.

In [58]:
print("`appearances`'s table structure:")

con.sql("""
    DESCRIBE appearances
""").show()

print("`appearances`'s table basic statistics:")

con.sql("""
    SUMMARIZE appearances
""").show()

`appearances`'s table structure:
┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ appearance_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ game_id                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_id              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_club_id         │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_current_club_id │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date                   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ player_name            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ competition_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL 

┌────────────────────────┬─────────────┬────────────────┬────────────────┬───────────────┬────────────────────────────┬──────────────────────┬────────────┬────────────┬────────────┬─────────┬─────────────────┐
│      column_name       │ column_type │      min       │      max       │ approx_unique │            avg             │         std          │    q25     │    q50     │    q75     │  count  │ null_percentage │
│        varchar         │   varchar   │    varchar     │    varchar     │     int64     │          varchar           │       varchar        │  varchar   │  varchar   │  varchar   │  int64  │  decimal(9,2)   │
├────────────────────────┼─────────────┼────────────────┼────────────────┼───────────────┼────────────────────────────┼──────────────────────┼────────────┼────────────┼────────────┼─────────┼─────────────────┤
│ appearance_id          │ VARCHAR     │ 2211607_111184 │ 4868900_996897 │       2015486 │ NULL                       │ NULL                 │ NULL       │ NULL

**Initial observations:**
- `appearances` has 13 columns and 1,894,350 rows, with no null values in any column.
- Dates run from 2012-07-03 to 2026-06-28, but valuations go back to 2000-01-20. 
    - Valuations before mid-2012 therefore have no match stats, which limits the usable modeling period.
- About 30,000 distinct players appear (approximate count), fewer than the 41,528 players with valuations, so some valued players have no appearances at all.
- `competition_id` has about 47 distinct values, including codes such as `AFCN` and `USC`, so domestic leagues, cups and international matches seem to be mixed. Season totals will therefore combine competitions.
- `minutes_played` ranges from 0 to 148 (median 90, mean about 68.5). A maximum of 148 is higher than a match with extra time normally allows, and rows with 0 minutes exist.
- Cards, goals and assists have plausible ranges (yellow cards 0 to 2, red cards 0 to 1, goals and assists 0 to 6).
- `player_current_club_id` is a VARCHAR with a minimum of -1. Like the current-club columns in `players`, it describes the player's club at the snapshot date, so it is excluded.
- `player_name` duplicates `players.name` and is not needed.

**Follow-ups:**
- Check for duplicate (`game_id`, `player_id`) pairs and for players that do not have an entry in `players`.
- Count rows with `minutes_played` above 120 and equal to 0, then decide how to handle them.
- Measure coverage: the share of valuations with at least one appearance in the prior 365 days, by year. This sets the start of the modeling period.
- Count valued players with no appearances at all.

#### Duplicate keys

First, we check whether every row in the table has a unique (`game_id`, `player_id`) pair and `appearance_id`.

In [59]:
con.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (game_id, player_id)) AS distinct_game_player_ids,
        COUNT(DISTINCT appearance_id) AS distinct_appearance_id
    FROM appearances
""").show()

┌────────────┬──────────────────────────┬────────────────────────┐
│ total_rows │ distinct_game_player_ids │ distinct_appearance_id │
│   int64    │          int64           │         int64          │
├────────────┼──────────────────────────┼────────────────────────┤
│    1894350 │                  1894350 │                1894350 │
└────────────┴──────────────────────────┴────────────────────────┘



**Conclusion:** Since both the (`game_id`, `player_id`) pair and the `appearance_id` are unique, each row in the appearance table represents a distinct player's performance in a specific game.

Next, we check whether any `player_id` in `appearances` is missing from `players`, and vice versa.

In [60]:
con.sql("""
    WITH app_players AS (
        SELECT DISTINCT player_id FROM appearances
    )
    SELECT
        (a.player_id IS NULL) AS no_appearances,
        (p.player_id IS NULL) AS not_in_players,
        COUNT(*) AS n_players
    FROM app_players a
    FULL OUTER JOIN players p ON a.player_id = p.player_id
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show()

┌────────────────┬────────────────┬───────────┐
│ no_appearances │ not_in_players │ n_players │
│    boolean     │    boolean     │   int64   │
├────────────────┼────────────────┼───────────┤
│ false          │ false          │     29530 │
│ false          │ true           │         1 │
│ true           │ false          │     20619 │
└────────────────┴────────────────┴───────────┘



**Observations:**
- 29,530 players appear in both tables.
- 1 player appears in `appearances` but not in `players` (an orphan row).
- 20,619 players in `players` (41.1%) have no appearances. This is expected, since `appearances` only covers matches from 2012 onwards and a limited set of competitions.

**Decision:**
- The orphan player cannot be linked to any player attributes (birth date, position, height), so their appearances will be removed.
- Players without appearances are handled in the next step, where we check how many of them have valuations.

#### Irregular playing minutes

Next, we look at the playing minutes of each appearance, focusing on irregular values: minutes above what a match normally allows (a league match lasts 90 minutes, and a knockout match with extra time lasts 120 minutes, plus stoppage time), and appearances with 0 minutes.

In [61]:
con.sql("""
    SELECT *
    FROM appearances a
    WHERE (a.minutes_played > 120) OR (a.minutes_played = 0)
""").show()

┌─────────────────┬─────────┬───────────┬────────────────┬────────────────────────┬────────────┬─────────────────────┬────────────────┬──────────────┬───────────┬───────┬─────────┬────────────────┐
│  appearance_id  │ game_id │ player_id │ player_club_id │ player_current_club_id │    date    │     player_name     │ competition_id │ yellow_cards │ red_cards │ goals │ assists │ minutes_played │
│     varchar     │  int32  │   int32   │     int32      │        varchar         │    date    │       varchar       │    varchar     │    int32     │   int32   │ int32 │  int32  │     int32      │
├─────────────────┼─────────┼───────────┼────────────────┼────────────────────────┼────────────┼─────────────────────┼────────────────┼──────────────┼───────────┼───────┼─────────┼────────────────┤
│ 2901276_296366  │ 2901276 │    296366 │           2477 │ 53646                  │ 2018-02-21 │ Oleksiy Gutsulyak   │ UKR1           │            0 │         0 │     0 │       0 │            135 │
│ 2901276_

**Observations:**
- Only 6 of 1,894,350 appearances (about 0.0003%) have irregular minutes: 3 with more than 120 minutes (135, 135 and 148) and 3 with 0 minutes.
- Two of the long ones belong to the same Ukrainian league game (`UKR1`, February 21st, 2018), where 135 minutes is not possible in regular play, so they are probably data errors. The third is a Copa del Rey match (`CDR`, December 4th, 2024) with 148 minutes, which is longer than extra time plus stoppage normally allows.
- The 3 zero-minute rows are all from 2026 (February to April), in the Dutch, French and Turkish leagues.

**Decision:** Drop all 6 rows. Their impact is negligible, and a row with 0 minutes should not count as an appearance in the season totals.|

#### Players with zero appearances

As mentioned, we look at players without appearances. They make up a sizeable share of `players` (20,619, or 41.1%), so we first check how many of them have valuations, since only valued players enter the modeling set.

In [62]:
con.sql("""
    WITH app_players AS (
        SELECT DISTINCT player_id
        FROM appearances
    )
    SELECT
        (a.player_id IS NULL) AS no_appearances,
        COUNT(DISTINCT v.player_id) AS n_players,
        COUNT(*) AS n_valuations,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_valuations,
        ROUND(COUNT(*)::DOUBLE / COUNT(DISTINCT v.player_id), 1) AS avg_valuations_per_player,
        MEDIAN(v.market_value_in_eur) AS median_value
    FROM player_valuations v
        LEFT JOIN app_players a USING (player_id)
    GROUP BY 1
""").show()

┌────────────────┬───────────┬──────────────┬────────────────┬───────────────────────────┬──────────────┐
│ no_appearances │ n_players │ n_valuations │ pct_valuations │ avg_valuations_per_player │ median_value │
│    boolean     │   int64   │    int64     │     double     │          double           │    double    │
├────────────────┼───────────┼──────────────┼────────────────┼───────────────────────────┼──────────────┤
│ false          │     28328 │       518863 │          79.06 │                      18.3 │     600000.0 │
│ true           │     13200 │       137438 │          20.94 │                      10.4 │     250000.0 │
└────────────────┴───────────┴──────────────┴────────────────┴───────────────────────────┴──────────────┘



**Observations:**
- 13,200 of 41,528 valued players (31.8%) have no appearances. They account for 137,438 of 656,301 valuations (20.9%).
- Compared with players who have appearances, they have fewer valuations per player (10.4 vs. 18.3 on average) and a lower median valuation (250,000 euros vs. 600,000 euros), suggesting a lower-profile group.

**Decision:** Valuations of players without appearances are dropped, since no match statistics can be built for them. This removes about one-fifth of the valuations, mostly lower-valued ones.

**Effect on the dataset:** Dropping players without appearances removes 13,200 players and 137,438 valuations (20.94%), leaving 28,328 players and 518,863 valuations. The remaining data skews toward higher-valued players in covered competition.

#### Inactive players

To measure how much match history each valuation has, we build a temporary table `window_stats` with the number of appearances in the 365 days before each valuation date, for players with at least one appearance.

In [63]:
con.sql("""
    CREATE OR REPLACE TEMP TABLE window_stats AS
    WITH players_with_appearances AS (
        SELECT DISTINCT player_id FROM appearances
    ),
    valuations_with_appearances AS (
        SELECT v.player_id, v.date
        FROM player_valuations v
        JOIN players_with_appearances pwa USING (player_id)
    )
    SELECT
        vwa.player_id,
        vwa.date,
        COUNT(a.game_id) AS n_apps
    FROM valuations_with_appearances vwa
    LEFT JOIN appearances a
        ON  a.player_id = vwa.player_id
        AND a.date BETWEEN vwa.date - 365 AND vwa.date - 1
    GROUP BY vwa.player_id, vwa.date
""").show()

┌────────┐
│ Count  │
│ int64  │
├────────┤
│ 518863 │
└────────┘



**Observations:** It has 518,863 rows, matching the valuations of players with appearances found earlier.

Next, we check by year how many of these valuations have at least one appearance in the window.

In [64]:
con.sql("""
    SELECT
        YEAR(w.date) AS "year",
        COUNT(*) AS "n_valuations",
        ROUND(100.0 * AVG((w.n_apps > 0)::INT), 2) AS "pct_with_apps",
        MEDIAN(w.n_apps) AS "median_apps"
    FROM window_stats w
    GROUP BY 1
    ORDER BY 1
""").df()

,year,n_valuations,pct_with_apps,median_apps
0,2000,1,0.00,0.0
1,2001,1,0.00,0.0
2,2003,2,0.00,0.0
3,2004,2224,0.00,0.0
4,2005,2362,0.00,0.0
5,2006,2578,0.00,0.0
6,2007,5439,0.00,0.0
7,2008,8945,0.00,0.0
8,2009,11504,0.00,0.0
9,2010,12983,0.00,0.0


**Observations:**
- Coverage is 0% before 2012, because `appearances` starts on 2012-07-03, and only 7.3% in 2012. A full 365-day window requires valuations from 2013-07-03 onward.
- From 2013, 42-82% of valuations have at least one appearance in the prior year, so an empty window is common even for players who have appearances.

Finally, we measure how much data remains after keeping only valuations from 2013-07-03 onward with at least one appearance in the window.

In [65]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE date >= DATE '2013-07-03' AND n_apps > 0)  AS n_valuations,
        ROUND(100.0 * COUNT(*) FILTER (WHERE date >= DATE '2013-07-03' AND n_apps > 0) / COUNT(*), 2) AS pct_valuations,
        COUNT(DISTINCT player_id) FILTER (WHERE date >= DATE '2013-07-03' AND n_apps > 0) AS n_players,
        ROUND(100.0 * COUNT(DISTINCT player_id) FILTER (WHERE date >= DATE '2013-07-03' AND n_apps > 0)
            / COUNT(DISTINCT player_id), 2) AS pct_players
    FROM window_stats
""").show()

┌──────────────┬────────────────┬───────────┬─────────────┐
│ n_valuations │ pct_valuations │ n_players │ pct_players │
│    int64     │     double     │   int64   │   double    │
├──────────────┼────────────────┼───────────┼─────────────┤
│       220291 │          42.46 │     26814 │       94.66 │
└──────────────┴────────────────┴───────────┴─────────────┘



**Observations:**
- 220,291 valuations remain, 42.46% of the 518,863 valuations of players with appearances, and about 33.6% of the original 656,301.
- 26,814 players remain, about 94.7% of the 28,328 players with appearances, and about 64.6% of the 41,528 valued players.
- The filter therefore removes many valuations but few players. The average falls from 18.3 to about 8.2 valuations per player (220,291 / 26,814).

**Decision:** The modeling period starts on 2013-07-03, and valuations with no appearances in the prior 365 days are dropped, since no stats can be computed and an empty window cannot be told apart from missing competition coverage. The remaining dataset has 220,291 valuations for 26,814 players. This is smaller than the original table, but still large enough for the model.

### Conclusion

#### Summary

Having profiled the four tables used in this project (`player_valuations`, `players`, `appearances`, and the derived 365-day appearance window), we now apply all cleaning rules together and measure the final dataset.

In [66]:
con.sql("""
    SELECT 
        COUNT(*) AS n_valuations,
        COUNT(DISTINCT v.player_id) AS n_players
    FROM player_valuations v
    JOIN players p
        ON p.player_id = v.player_id
    JOIN window_stats w
        ON w.player_id = v.player_id AND w.date = v.date
    WHERE v.market_value_in_eur > 0
        AND p.position <> 'Missing'
        AND p.date_of_birth IS NOT NULL
        AND date_sub('year', p.date_of_birth, v.date) >= 15
        AND v.date >= DATE '2013-07-03'
        AND w.n_apps > 0
""").show()

┌──────────────┬───────────┐
│ n_valuations │ n_players │
│    int64     │   int64   │
├──────────────┼───────────┤
│       220008 │     26746 │
└──────────────┴───────────┘



#### Interim dataset rule

**Grain and target:** one row per player per valuation date. The target is `market_value_in_eur` from `player_valuations`.

**Cleaning rules:**

| No. | Step | Affected | Rationale |
|:---:|---|:---:|---|
| 1 | Drop valuations of €0 | 1 valuation | Likely an entry error. |
| 2 | Drop valuations with no date of birth | 600 valuations | Age at valuation is a core feature and can't be worked out without a birth date. |
| 3 | Drop valuations where age is under 15 | 12 valuations | Likely data errors (i.e., ages 2, 3, 11). |
| 4 | Set heights under 150 cm to missing | 13 players | Physically implausible; these are imputed later rather than dropped |
| 5 | Recode missing `foot` as `unknown` | 5,893 players | `foot` is a core feature, so missing values become their own category instead of being dropped |
| 6 | Drop players with no appearances | 13,200 players (137,438 valuations) | Match statistics can't be built.|
| 7 | Drop players whose position is `Missing` | 129 players (891 valuations) | Players without a position can't be put in a position group |
| 8 | Keep only valuations dated on or after 2013-07-03 that have at least one appearance in the prior 365 days | 298,572 valuations dropped | `appearances` starts on 2012-07-03, so 2013-07-03 is the first date with a full 365-day window.|
| 9 | Drop appearances with irregular minutes | 6 appearances (3 over 120 min, 3 at 0 min) | Likely data errors; a 0-minute row isn't a real appearance |

**Excluded columns:**
- `market_value_in_eur` and `highest_market_value_in_eur` in `players`: identical to the latest and peak valuations, which would leak the target.
- `current_club_*`, `player_club_domestic_competition_id` and `player_current_club_id`: snapshot values that do not match the club at the valuation date.
- `sub_position`, nationality and birthplace columns, names and URLs.
- The `clubs`, `competitions` and `transfers` tables are not used.

#### Clean-up

After the overview is complete, we close the connection to the database.

In [67]:
con.close()